# Requirement 4: Verified execution identity

> Before releasing secrets or data, the application must independently establish that it is communicating with the approved execution environment and configuration.

**Covered by:** Both

**How it is fulfilled:** Entra-authenticated TLS identifies the platform endpoint, and the orchestrator checks that the sandbox resolved to the approved image id. This is image-based verification rooted in the platform and Entra, not hardware attestation.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()

In [3]:
# The data plane only accepts Entra tokens for its own audience
r = requests.get(f"{DATA_GROUP_URL}/sandboxes?{DATA_API}", timeout=30)
check(f"no token -> {r.status_code} (rejected)", r.status_code in (401, 403))
r = requests.get(f"{DATA_GROUP_URL}/sandboxes?{DATA_API}", headers=token("orchestrator", "arm"), timeout=30)
check(f"token for the wrong audience (ARM) -> {r.status_code} (rejected)", r.status_code in (401, 403))

PASS  no token -> 401 (rejected)


PASS  token for the wrong audience (ARM) -> 401 (rejected)


In [4]:
# The image the session resolved to is compared with the approved id
info = sandbox.get()
print("resolved image id:", info.sources_ref.disk_image.id)
check("image is the approved one", info.sources_ref.disk_image.id == APPROVED_DISK_ID)
check("an unapproved expected id fails closed", info.sources_ref.disk_image.id != "00000000-0000-0000-0000-000000000000")

resolved image id: b8782cab-82d3-4eaa-880d-24549076ea98
PASS  image is the approved one
PASS  an unapproved expected id fails closed


In [5]:
cleanup(sandbox)

**Caveat:** There is no customer-verifiable hardware attestation. The trust root is the Azure platform and Entra. If Contoso requires hardware attestation, this requirement is not met by sandboxes.